In [1]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [2]:
file_path = "/home/kobugi/papaya/ece-5464/project_3/data/K8.xlsx"

In [3]:
df = pd.read_excel(file_path, header= None)

In [4]:
df.rename(columns={df.columns[-1]: "result"}, inplace=True)

In [5]:
question_marks = df.map(lambda x: x == "?")
question_mark_count = question_marks.sum()
print("Number of '?' values per column:\n", question_mark_count)

Number of '?' values per column:
 0         180
1         180
2         180
3         180
4         180
         ... 
5404       57
5405       57
5406       57
5407       57
result      0
Length: 5409, dtype: int64


In [6]:
df = df[df[4826] != "?"]

In [7]:
df = df.replace("?", np.nan)

/tmp/ipykernel_53351/3407341085.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace("?", np.nan)


In [8]:
x_df = df.drop(columns=["result"])  
y = df["result"] 

In [9]:
from sklearn.impute import KNNImputer
x_df = x_df.applymap(lambda x: np.nan if pd.isna(x) else x)
x_df = x_df.astype(float)
imputer = KNNImputer(n_neighbors=5)
x_df = imputer.fit_transform(x_df)  
x_df = pd.DataFrame(x_df)
print(x_df.isna().sum().sum())

/tmp/ipykernel_53351/2054322235.py:2: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  x_df = x_df.applymap(lambda x: np.nan if pd.isna(x) else x)


0


In [10]:
print(x_df.shape, y.shape)

(16715, 5408) (16715,)


In [11]:
print(x_df.shape)
print(y.shape)

(16715, 5408)
(16715,)


In [12]:
y = y.squeeze()

In [13]:
x_df = x_df.reset_index(drop=True)
y = y.reset_index(drop=True)

In [14]:
concat_df = pd.concat([x_df, y], axis=1)

In [15]:
concat_df.head()

,0,1,2,3,4,5,6,7,8,9,...,5399,5400,5401,5402,5403,5404,5405,5406,5407,result
0,-0.161,-0.014,0.002,-0.036,-0.033,-0.093,0.025,0.005,0.000,-0.015,...,0.006,0.013,0.021,0.020,0.016,-0.011,0.003,0.010,-0.007,inactive
1,-0.158,-0.002,-0.012,-0.025,-0.012,-0.106,0.013,0.005,0.000,-0.002,...,0.002,-0.008,0.007,0.015,-0.008,-0.011,-0.004,0.013,0.005,inactive
2,-0.169,-0.025,-0.010,-0.041,-0.045,-0.069,0.038,0.014,0.008,-0.014,...,0.019,0.010,0.025,0.025,0.021,-0.012,0.006,0.016,-0.018,inactive
3,-0.183,-0.051,-0.023,-0.077,-0.092,-0.015,0.071,0.027,0.020,-0.019,...,0.051,0.012,0.050,0.038,0.051,-0.015,0.017,0.027,-0.049,inactive
4,-0.154,0.005,-0.011,-0.013,-0.002,-0.115,0.005,0.002,-0.003,0.002,...,-0.011,0.012,0.009,0.003,-0.001,0.002,-0.006,0.009,0.013,inactive


In [16]:
df = concat_df

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split


X = x_df.copy()  

weights_list = ["uniform", "distance"]
neighbors_list = [1, 3, 5, 9, 11, 15, 21]
dataset_types = ["unbalanced", "balanced"]


results = []


for dataset_type in dataset_types:
    if dataset_type == "unbalanced":
        X_current = X.copy()
        y_current = y.copy()
    else:
        balanced_df = df.copy()
        min_count = balanced_df["result"].value_counts().min()
        balanced_df = balanced_df.groupby("result", group_keys=False).apply(
            lambda grp: grp.sample(n=min_count, random_state=42)
        ).reset_index(drop=True)

        X_current = balanced_df.drop(columns=["result"])
        y_current = balanced_df["result"]
    print(X_current.shape, y_current.shape)

    X_train, X_test, y_train, y_test = train_test_split(
        X_current, y_current, test_size=0.3, random_state=22222, stratify=y_current
    )

    for weight in weights_list:
        for n in neighbors_list:

            neigh = KNeighborsClassifier(n_neighbors=n, weights=weight,n_jobs=-1)
            neigh.fit(X_train, y_train)


            train_score = neigh.score(X_train, y_train)
            test_score = neigh.score(X_test, y_test)


            results.append({
                "dataset": dataset_type,
                "weight": weight,
                "n_neighbors": n,
                "train_score": train_score,
                "test_score": test_score
            })


results_df = pd.DataFrame(results)
print(results_df)


(16715, 5408) (16715,)


In [ ]:
import pandas as pd
from tabulate import tabulate  
print(tabulate(results_df, headers="keys", tablefmt="pretty"))

In [ ]:
best_test = results_df.loc[results_df["test_score"].idxmax()]
print("Best Test Performance:")
print(best_test)

In [ ]:
results_df["diff"] = abs(results_df["train_score"] - results_df["test_score"])

largest_diff = results_df.loc[results_df["diff"].idxmax()]
smallest_diff = results_df.loc[results_df["diff"].idxmin()]

print("Largest Train-Test Difference:")
print(largest_diff)

print("\nSmallest Train-Test Difference:")
print(smallest_diff)